In [1]:
# Importações da exploração aleatoria

from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
from sklearn.model_selection import GroupKFold as KFold
from sklearn.svm import SVC

In [2]:
uri = "https://gist.githubusercontent.com/guilhermesilveira/e99a526b2e7ccc6c3b70f53db43a87d2/raw/1605fc74aa778066bf2e6695e24d53cf65f2f447/machine-learning-carros-simulacao.csv"
dados = pd.read_csv(uri).drop(columns=["Unnamed: 0"], axis=1)
dados.head()

,preco,vendido,idade_do_modelo,km_por_ano
0,30941.02,1,18,35085.22134
1,40557.96,1,20,12622.05362
2,89627.50,0,12,11440.79806
3,95276.14,0,3,43167.32682
4,117384.68,1,4,12770.11290


In [3]:
# situacao de azar onde as classes são treinadas por padrão

dados_azar = dados.sort_values("vendido", ascending=True)
x_azar = dados_azar[["preco", "idade_do_modelo","km_por_ano"]]
y_azar = dados_azar["vendido"]
dados_azar.head()

,preco,vendido,idade_do_modelo,km_por_ano
4999,74023.29,0,12,24812.80412
5322,84843.49,0,13,23095.63834
5319,83100.27,0,19,36240.72746
5316,87932.13,0,16,32249.56426
5315,77937.01,0,15,28414.50704


In [4]:
SEED=301
np.random.seed(SEED)

dados['modelo'] = dados.idade_do_modelo + np.random.randint(-2, 3, size=10000)
dados.modelo = dados.modelo + abs(dados.modelo.min()) + 1
dados.head()

espaco_de_parametros = {
    "max_depth" : [3, 5],
    "min_samples_split": [32, 64, 128],
    "min_samples_leaf": [32, 64, 128],
    "criterion": ["gini", "entropy"]

}

modelo = SVC()

busca = RandomizedSearchCV(DecisionTreeClassifier(),
                    espaco_de_parametros,
                    n_iter = 16,
                    cv = KFold(n_splits = 5, shuffle=True),
                    random_state=SEED)

busca.fit(x_azar, y_azar, groups = dados.modelo)
resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.023806,0.004525,0.006904,0.002137,128,128,5,gini,"{'min_samples_split': 128, 'min_samples_leaf':...",0.772936,0.778880,0.787048,0.794697,0.785662,0.783845,0.007416,13
1,0.017065,0.002022,0.003600,0.000919,64,32,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.778287,0.778880,0.787048,0.796847,0.785662,0.785345,0.006737,1
2,0.013616,0.002485,0.003287,0.000738,64,128,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.778287,0.778880,0.787048,0.796847,0.785662,0.785345,0.006737,1
3,0.028699,0.010470,0.005190,0.002962,32,64,5,entropy,"{'min_samples_split': 32, 'min_samples_leaf': ...",0.778287,0.776401,0.787048,0.792906,0.785662,0.784061,0.006029,10
4,0.026019,0.007243,0.005449,0.001733,64,64,5,entropy,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.778287,0.776401,0.787048,0.792906,0.785662,0.784061,0.006029,10


In [5]:
print(dados.columns)

Index(['preco', 'vendido', 'idade_do_modelo', 'km_por_ano', 'modelo'], dtype='object')


In [6]:
np.random.seed(SEED)
dados['modelo'] = dados.idade_do_modelo + np.random.randint(-2, 3, size=10000)
dados.modelo = dados.modelo + abs(dados.modelo.min()) + 1
dados.head()

,preco,vendido,idade_do_modelo,km_por_ano,modelo
0,30941.02,1,18,35085.22134,18
1,40557.96,1,20,12622.05362,24
2,89627.50,0,12,11440.79806,14
3,95276.14,0,3,43167.32682,6
4,117384.68,1,4,12770.11290,5


In [7]:
print(dados.modelo)

0       18
1       24
2       14
3        6
4        5
        ..
9995    14
9996    17
9997     6
9998    11
9999    23
Name: modelo, Length: 10000, dtype: int64


In [8]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(modelo, x_azar, y_azar, groups=dados.modelo, cv = KFold(n_splits=2, shuffle=True))
scores

array([0.76322548, 0.7757943 ])

In [9]:
def imprime_score(scores):
  media = scores.mean() * 100
  desvio = scores.std() * 100
  print("Accuracy médio %.2f" % media)
  print("Intervalo [%.2f, %.2f]" % (media - 2 * desvio, media + 2 * desvio))

In [10]:
melhor = busca.best_estimator_
print(melhor)

DecisionTreeClassifier(max_depth=3, min_samples_leaf=32, min_samples_split=64)


In [11]:
# Customizando espaço de hiperparametros

from scipy.stats import randint


SEED=301
np.random.seed(SEED)


espaco_de_parametros = {
    "max_depth" : [3, 5, 10, 15, 20, 30, None],
    "min_samples_split": randint(32,128),
    "min_samples_leaf": randint(32,128),
    "criterion": ["gini", "entropy"]

}

busca = RandomizedSearchCV(DecisionTreeClassifier(),
                    espaco_de_parametros,
                    n_iter = 16,
                    cv = KFold(n_splits = 5, shuffle=True),
                    random_state=SEED)

busca.fit(x_azar, y_azar, groups = dados.modelo)
resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.029617,0.008212,0.008118,0.002912,entropy,3,71,100,"{'criterion': 'entropy', 'max_depth': 3, 'min_...",0.776574,0.787289,0.782801,0.795190,0.789038,0.786179,0.006233,1
1,0.046788,0.005465,0.007510,0.000950,gini,15,93,111,"{'criterion': 'gini', 'max_depth': 15, 'min_sa...",0.758971,0.786921,0.782801,0.785493,0.785936,0.780024,0.010615,12
2,0.045980,0.004196,0.007857,0.000608,gini,20,124,88,"{'criterion': 'gini', 'max_depth': 20, 'min_sa...",0.773189,0.786187,0.781472,0.785105,0.788004,0.782791,0.005254,4
3,0.052787,0.008416,0.007171,0.002001,gini,None,46,62,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.756940,0.778472,0.770833,0.777347,0.781799,0.773078,0.008819,15
4,0.041695,0.007326,0.007263,0.000414,gini,15,126,84,"{'criterion': 'gini', 'max_depth': 15, 'min_sa...",0.773189,0.786187,0.779699,0.785493,0.786970,0.782307,0.005232,5


In [12]:
scores = cross_val_score(modelo, x_azar, y_azar, groups=dados.modelo, cv = KFold(n_splits=2, shuffle=True))
imprime_score(scores)

Accuracy médio 77.41
Intervalo [77.11, 77.71]


# Imprimenindo resultados de mean test score depois de rodar de forma aleatoria utilizando random grid

In [13]:
resultados_ordenados_media_teste_score = resultados.sort_values('mean_test_score', ascending=False)

for indice, linha in resultados_ordenados_media_teste_score.iterrows():
    print("%.3f +-(%.3f) %s" % (linha.mean_test_score, linha.std_test_score*2, linha.params))

0.786 +-(0.012) {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 71, 'min_samples_split': 100}
0.785 +-(0.012) {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 64, 'min_samples_split': 67}
0.785 +-(0.010) {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 73, 'min_samples_split': 72}
0.783 +-(0.011) {'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 124, 'min_samples_split': 88}
0.782 +-(0.010) {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 126, 'min_samples_split': 84}
0.782 +-(0.014) {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 125, 'min_samples_split': 59}
0.781 +-(0.018) {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 101, 'min_samples_split': 52}
0.781 +-(0.018) {'criterion': 'gini', 'max_depth': 30, 'min_samples_leaf': 100, 'min_samples_split': 84}
0.781 +-(0.020) {'criterion': 'gini', 'max_depth': 15, 'min_samples_leaf': 103, 'min_samples_split': 96}
0.781 +-(0.018) {'criterion': 'gini', 'max_depth': 1

# Exploração mais a funda de forma aleatoria

In [14]:
SEED=301
np.random.seed(SEED)


espaco_de_parametros = {
    "max_depth" : [3, 5],
    "min_samples_split": [32, 64, 128],
    "min_samples_leaf": [32, 64, 128],
    "criterion": ["gini", "entropy"]

}

busca = RandomizedSearchCV(DecisionTreeClassifier(),
                    espaco_de_parametros,
                    n_iter = 64,
                    cv = KFold(n_splits = 5, shuffle=True),
                    random_state=SEED)

busca.fit(x_azar, y_azar, groups = dados.modelo)
resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

c:\Users\lara-\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 36 is smaller than n_iter=64. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_min_samples_split,param_min_samples_leaf,param_max_depth,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.026663,0.004609,0.007561,0.002888,32,32,3,gini,"{'min_samples_split': 32, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.79519,0.789038,0.786179,0.006233,1
1,0.022867,0.004829,0.006509,0.000673,64,32,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.79519,0.789038,0.786179,0.006233,1
2,0.024271,0.001413,0.007170,0.001004,128,32,3,gini,"{'min_samples_split': 128, 'min_samples_leaf':...",0.776574,0.787289,0.782801,0.79519,0.789038,0.786179,0.006233,1
3,0.027532,0.002751,0.007638,0.001225,32,64,3,gini,"{'min_samples_split': 32, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.79519,0.789038,0.786179,0.006233,1
4,0.030331,0.004899,0.008904,0.002942,64,64,3,gini,"{'min_samples_split': 64, 'min_samples_leaf': ...",0.776574,0.787289,0.782801,0.79519,0.789038,0.786179,0.006233,1


In [15]:
scores = cross_val_score(modelo, x_azar, y_azar, groups=dados.modelo, cv = KFold(n_splits=2, shuffle=True))
imprime_score(scores)

Accuracy médio 77.29
Intervalo [75.76, 78.82]


In [16]:
resultados_ordenados_media_teste_score = resultados.sort_values('mean_test_score', ascending=False)

for indice, linha in resultados_ordenados_media_teste_score.iterrows():
    print("%.3f +-(%.3f) %s" % (linha.mean_test_score, linha.std_test_score*2, linha.params))

0.786 +-(0.012) {'min_samples_split': 32, 'min_samples_leaf': 32, 'max_depth': 3, 'criterion': 'gini'}
0.786 +-(0.012) {'min_samples_split': 64, 'min_samples_leaf': 32, 'max_depth': 3, 'criterion': 'gini'}
0.786 +-(0.012) {'min_samples_split': 128, 'min_samples_leaf': 128, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 64, 'min_samples_leaf': 128, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 32, 'min_samples_leaf': 128, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 128, 'min_samples_leaf': 64, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 64, 'min_samples_leaf': 64, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 32, 'min_samples_leaf': 64, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 128, 'min_samples_leaf': 32, 'max_depth': 3, 'criterion': 'entropy'}
0.786 +-(0.012) {'min_samples_split': 64, 'min

# Comparando GridSearchCV e RandomizedSearch

In [19]:
# RandomForestClassifier é um tipo de arvore do Sklearn que faz analises e classificações a partir de mais de uma arvore diferente do classficador decisiontree que utilzia apenas uma arvore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
import time

SEED = 301
np.random.seed(SEED)


espaco_de_parametros = {
    "n_estimators":[10,101],
    "bootstrap":[True, False],
    "max_depth" : randint(3,6),
    "min_samples_leaf": randint(32,129),
    "min_samples_split": randint(32,129),
    "criterion":["gini",  "entropy"]
}

tic = time.time()
busca = RandomizedSearchCV(
                RandomForestClassifier(),
                espaco_de_parametros,
                n_iter = 5,
                cv = StratifiedShuffleSplit(n_splits=1, test_size=0.2))

busca.fit(x_azar, y_azar, groups=dados.modelo)
tac = time.time()


tempo_que_passou = tic - tac
print(tempo_que_passou)


resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

c:\Users\lara-\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:2430: UserWarning: The groups parameter is ignored by StratifiedShuffleSplit
  warnings.warn(


-1.4628937244415283


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_bootstrap,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_n_estimators,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
0,0.327673,0.0,0.012916,0.0,False,gini,3,50,93,101,"{'bootstrap': False, 'criterion': 'gini', 'max...",0.7750,0.7750,0.0,1
1,0.029231,0.0,0.003351,0.0,False,gini,3,124,88,10,"{'bootstrap': False, 'criterion': 'gini', 'max...",0.7470,0.7470,0.0,5
2,0.028459,0.0,0.002326,0.0,True,gini,5,56,126,10,"{'bootstrap': True, 'criterion': 'gini', 'max_...",0.7645,0.7645,0.0,4
3,0.260050,0.0,0.011518,0.0,True,entropy,3,52,80,101,"{'bootstrap': True, 'criterion': 'entropy', 'm...",0.7750,0.7750,0.0,1
4,0.321143,0.0,0.014146,0.0,False,entropy,3,120,70,101,"{'bootstrap': False, 'criterion': 'entropy', '...",0.7750,0.7750,0.0,1


In [ ]:
resultados_ordenados_pela_media = resultados.sort_values("mean_test_score", ascending=False)
for indice, linha in resultados_ordenados_pela_media[:5].iterrows():
  print("%.3f +-(%.3f) %s" % (linha.mean_test_score, linha.std_test_score*2, linha.params))

0.781 +-(0.015) {'bootstrap': False, 'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 32, 'min_samples_split': 64, 'n_estimators': 10}
0.781 +-(0.014) {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 32, 'min_samples_split': 64, 'n_estimators': 10}
0.779 +-(0.017) {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 32, 'min_samples_split': 128, 'n_estimators': 100}
0.779 +-(0.010) {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 64, 'min_samples_split': 64, 'n_estimators': 10}
0.778 +-(0.014) {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 32, 'min_samples_split': 32, 'n_estimators': 100}


In [ ]:
tic = time.time()
scores = cross_val_score(modelo, x_azar, y_azar, groups=modelo, cv = KFold(n_splits=5, shuffle=True))
tac = time.time()
tempo_passado = tac - tic
print("Tempo %.2f segundos" % tempo_passado)

imprime_score(scores)
melhor = busca.best_estimator_
print(melhor)

InvalidParameterError: The 'groups' parameter of cross_val_score must be an array-like or None. Got SVC() instead.

# Se n conseguir usar o cross validation

In [20]:
# RandomForestClassifier é um tipo de arvore do Sklearn que faz analises e classificações a partir de mais de uma arvore diferente do classficador decisiontree que utilzia apenas uma arvore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
import time

SEED = 301
np.random.seed(SEED)


espaco_de_parametros = {
    "n_estimators":[10,101],
    "bootstrap":[True, False],
    "max_depth" : randint(3,6),
    "min_samples_leaf": randint(32,129),
    "min_samples_split": randint(32,129),
    "criterion":["gini",  "entropy"]
}

tic = time.time()
busca = RandomizedSearchCV(
                RandomForestClassifier(),
                espaco_de_parametros,
                n_iter = 5,
                cv = StratifiedShuffleSplit(n_splits=1, test_size=0.2))

busca.fit(x_azar, y_azar, groups=dados.modelo)
tac = time.time()


tempo_que_passou = tic - tac
print(tempo_que_passou)


resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

c:\Users\lara-\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:2430: UserWarning: The groups parameter is ignored by StratifiedShuffleSplit
  warnings.warn(


-1.6408088207244873


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_bootstrap,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_n_estimators,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
0,0.276879,0.0,0.014957,0.0,False,gini,3,50,93,101,"{'bootstrap': False, 'criterion': 'gini', 'max...",0.7750,0.7750,0.0,1
1,0.035233,0.0,0.002587,0.0,False,gini,3,124,88,10,"{'bootstrap': False, 'criterion': 'gini', 'max...",0.7470,0.7470,0.0,5
2,0.031964,0.0,0.002583,0.0,True,gini,5,56,126,10,"{'bootstrap': True, 'criterion': 'gini', 'max_...",0.7645,0.7645,0.0,4
3,0.305053,0.0,0.010804,0.0,True,entropy,3,52,80,101,"{'bootstrap': True, 'criterion': 'entropy', 'm...",0.7750,0.7750,0.0,1
4,0.418178,0.0,0.019796,0.0,False,entropy,3,120,70,101,"{'bootstrap': False, 'criterion': 'entropy', '...",0.7750,0.7750,0.0,1


In [23]:
from sklearn.model_selection import train_test_split

SEED = 301
np.random.seed(SEED) 

x_treino_teste, x_validacao, y_treino_teste, y_validacao = train_test_split(x_azar, y_azar, test_size=0.2, shuffle=True, stratify=y_azar)

print(x_treino_teste.shape)
print(x_validacao.shape)
print(y_treino_teste.shape)
print(y_validacao.shape)

(8000, 3)
(2000, 3)
(8000,)
(2000,)


In [24]:
espaco_de_parametros = {
    "n_estimators" : randint(10, 101),
    "max_depth" : randint(3, 6),
    "min_samples_split": randint(32, 129),
    "min_samples_leaf": randint(32, 129),
    "bootstrap" : [True, False],
    "criterion": ["gini", "entropy"]

}

split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.25)

tic = time.time()
busca = RandomizedSearchCV(RandomForestClassifier(),
                    espaco_de_parametros,
                    n_iter = 5,
                    cv = split)
busca.fit(x_treino_teste, y_treino_teste)
tac = time.time()
tempo_que_passou = tac - tic
print("Tempo %.2f segundos" % tempo_que_passou)



resultados = pd.DataFrame(busca.cv_results_)
resultados.head()

Tempo 0.87 segundos


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_bootstrap,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split,param_n_estimators,params,split0_test_score,mean_test_score,std_test_score,rank_test_score
0,0.130396,0.0,0.011108,0.0,True,gini,5,89,58,24,"{'bootstrap': True, 'criterion': 'gini', 'max_...",0.7885,0.7885,0.0,3
1,0.077276,0.0,0.004143,0.0,False,entropy,3,71,55,27,"{'bootstrap': False, 'criterion': 'entropy', '...",0.7920,0.7920,0.0,1
2,0.057769,0.0,0.003866,0.0,True,entropy,3,33,99,21,"{'bootstrap': True, 'criterion': 'entropy', 'm...",0.7910,0.7910,0.0,2
3,0.154357,0.0,0.011029,0.0,True,gini,3,98,100,67,"{'bootstrap': True, 'criterion': 'gini', 'max_...",0.7785,0.7785,0.0,5
4,0.236409,0.0,0.008178,0.0,False,entropy,4,88,64,63,"{'bootstrap': False, 'criterion': 'entropy', '...",0.7885,0.7885,0.0,3


In [25]:
tic = time.time()
scores = cross_val_score(busca, x_validacao, y_validacao, cv = split)
tac = time.time()
tempo_passado = tac - tic
print("Tempo %.2f segundos" % tempo_passado)

scores

Tempo 1.44 segundos


array([0.732])